# NB01 — Do prompt ao agente

**Pergunta:** o que separa um prompt de um agente?  
**Caminho:** `v0` (agente sem ferramentas, com a política colada) → `v1` (ferramentas, decisão estruturada, estado, contexto enxuto, memória).

Cada seção mostra um problema real do RangoJá e a correção. As execuções vêm de **gravações do modelo real**; quando uma gravação foi escolhida entre várias, a célula diz a taxa.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender_caso`, `afirmar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, atender_caso, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, comparar, afirmar, esperar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, v0_so_prompt, v1_agente, v2_controlado, v3_robusto, montar
preparar_ambiente()

modo,replay
modelo,gpt-4o-mini
chave de API,não (use replay)
observabilidade,desligado (RANGOJA_PHOENIX=off)


## N1.1 — O prompt que funciona na demo

Antes de qualquer atalho, o agente inteiro, em ADK puro. Primeiro, o que chegou: a reclamação do cliente e o pedido como ele está no sistema.

In [2]:
mensagem = "Meu pedido 4821 chegou frio e faltou o refri."

pedido = dados.pedido("4821")
for item in pedido.itens:
    print(f"{item.nome:22s} R$ {float(item.preco):6.2f}   {'entregue' if item.entregue else 'NÃO entregue'}")
print(f"{'total':22s} R$ {float(pedido.total):6.2f}")

Hambúrguer artesanal   R$  38.00   entregue
Batata média           R$  18.00   entregue
Refrigerante 2L        R$   8.00   NÃO entregue
total                  R$  70.00


O agente é um `LlmAgent` do ADK: um nome, um modelo e uma instrução. Aqui a instrução é o prompt com a política inteira colada. O modelo vem do `.env`: em `live`, o `LiteLlm` chama o modelo de `RANGOJA_MODELO` com a sua chave; em `replay`, uma gravação do modelo real responde no lugar dele.

As linhas comentadas são os **pontos de intervenção**: os lugares onde o código pode agir sobre o agente. Cada seção do material preenche um deles.

In [3]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from rangoja.notebook import gravacao

politica = "\n\n".join(f"## {tema}\n{texto}" for tema, texto in dados.politica_secoes().items())
PROMPT = ("Você atende clientes do RangoJá, um app de delivery. Responda em português, com cordialidade e objetividade."
          "\n\nPolítica de atendimento:\n" + politica)

if config.modo() == "replay":
    llm = gravacao("N1.1/falha")  # sem chave: o que o modelo real respondeu quando gravamos
else:
    llm = LiteLlm(model=f"openai/{config.modelo_primario()}")  # o modelo de RANGOJA_MODELO (.env), com a sua chave

agente = LlmAgent(
    name="unico",
    model=llm,
    instruction=PROMPT,
    # tools=[buscar_pedido, registrar_reembolso, ...]   ← N1.2  ferramentas trazem fatos e agem
    # output_schema=Decisao                             ← N1.4  decisão que o código consegue ler
    # before_model_callback=guardrail_entrada           ← N2.2  filtra o que chega ao modelo
    # after_model_callback=guardrail_saida              ← N2.3  filtra o que sai para o cliente
    # before_tool_callback=limite_ferramenta            ← N2.4, N3.4  autoriza cada ação
    # on_tool_error_callback=erro_estruturado           ← N3.2  erro de ferramenta vira resultado
)
print(f"modelo: {'gravação N1.1/falha' if config.modo() == 'replay' else config.modelo_primario() + ' ao vivo'}")
print(f"instrução com {len(PROMPT)} caracteres; ferramentas: {len(agente.tools)}")

modelo: gravação N1.1/falha
instrução com 2054 caracteres; ferramentas: 0


Para rodar, o `Runner` do ADK conduz o loop: entrega a mensagem ao agente, chama o modelo e devolve **eventos**. A sessão guarda a conversa.

In [4]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService  # ← N3.5: SqliteSessionService sobrevive à queda do processo
from google.genai import types

runner = Runner(app_name="rangoja", agent=agente, session_service=InMemorySessionService())
sessao = await runner.session_service.create_session(app_name="rangoja", user_id="101")
pergunta = types.Content(role="user", parts=[types.Part(text=mensagem)])

async for evento in runner.run_async(user_id="101", session_id=sessao.id, new_message=pergunta):
    for parte in (evento.content.parts if evento.content else []):
        if parte.function_call:
            print(f"[{evento.author}] chama {parte.function_call.name}({dict(parte.function_call.args)})")
        if parte.text:
            print(f"[{evento.author}] {parte.text}")

[unico] Lamento pela experiência que você teve. Seu pedido apresenta duas questões: 

1. Qualidade: O alimento chegou frio.
2. Item faltando: O refrigerante não foi entregue.

Para resolver, posso oferecer um voucher de R$ 15 pela qualidade do alimento e fazer o reembolso do valor do refrigerante. Você gostaria de seguir dessa forma?


**Parece certo, e é esse o problema.** A resposta gravada segue a política (voucher de até R$ 15 por qualidade, reembolso do item que faltou). Mas olhe o que aconteceu de fato:

- **Nada foi consultado.** O agente acreditou no cliente: não tinha como saber se o refrigerante faltou mesmo, nem quanto custava.
- **Nada foi feito.** O cliente leu "reembolso", mas não existe reembolso no sistema: o agente tem zero ferramentas. Ao vivo, o modelo às vezes ainda diz que está "processando".
- **O código não tem o que executar.** A saída é texto: não há categoria nem valor que um sistema possa ler e cumprir.

Funciona na demo. Não funciona em produção.

Daqui em diante usamos um **atalho**: `atender_caso(caso, com=agente, replay=...)` faz o que as duas células acima fizeram (monta o agente e roda o `Runner`) e devolve o resultado em campos que o código pode verificar. O `replay=` diz qual resposta gravada usar em modo `replay`, sem chave. Em modo `live`, ele é ignorado: o modelo responde de verdade. O agente é uma **versão**: um conjunto nomeado de componentes. Esta é a `v0_so_prompt`:

In [5]:
from rangoja.versoes import v0_so_prompt

v0_so_prompt  # digitar o nome de uma versão mostra o que ela é

prompt,agente sem ferramentas; política na instrução


In [6]:
afirmar(v0_so_prompt.construir().agente_raiz.instruction == PROMPT, "v0_so_prompt é o agente montado acima (mesmo prompt)")

resultado = atender_caso("C01", com=v0_so_prompt, replay="N1.1/falha")  # C01: cliente 101, pedido 4821
# replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
print("decisão:             ", resultado.decisao)     # None: nada que o código possa ler
print("ferramentas chamadas:", resultado.trajetoria)  # []: o agente não consultou nem registrou nada
print("resposta veio de:    ", resultado.fonte)

decisão:              None
ferramentas chamadas: []
resposta veio de:     ['gravacao']


In [7]:
# 🎯 EXPERIMENTE: a ficha completa de uma versão, lida do agente construído (prompt, ferramentas, callbacks, execução)
# comparar(None, v0_so_prompt)

O modelo só sabe o que está no contexto. Texto plausível não é fato. Como dar fatos ao modelo? **Ferramentas.**

## N1.2 — Ferramentas trazem fatos

A mesma reclamação. Agora o agente ganha **ferramentas**: funções Python que o modelo pode pedir para executar, como `buscar_pedido` e `registrar_reembolso`. Primeiro, o que muda de um agente para o outro, lido dos próprios agentes:

In [8]:
sem_ferramentas = v0_so_prompt
com_ferramentas = v0_so_prompt.com("ferramentas")
comparar(sem_ferramentas, com_ferramentas)

Uma ferramenta é uma função Python comum. O modelo não vê o código: o ADK envia a ele só o **nome, a descrição (a docstring) e os parâmetros**, e o modelo decide se pede para chamá-la. É isto que ele recebe:

In [9]:
from google.adk.tools import FunctionTool

ferramentas_do_agente = {f.__name__: f for f in com_ferramentas.construir().agente_raiz.tools}
for nome, funcao in ferramentas_do_agente.items():
    print(f"{nome:22s} {funcao.__doc__}")

print("\no que o ADK envia ao modelo sobre buscar_pedido:")
print(FunctionTool(ferramentas_do_agente["buscar_pedido"])._get_declaration().model_dump_json(exclude_none=True, indent=1))

buscar_pedido          Consulta um pedido pelo número. Devolve itens, valores, pagamentos e atraso.
consultar_politica     Consulta um trecho da política de atendimento. Temas: categorias, item_faltando, item_errado, cobranca, atraso, qualidade, golpe, limites, recorrencia, privacidade, memoria, fora_do_escopo.
oferecer_cupom         Oferece um cupom de desconto ao cliente. Até R$ 15 por atendimento.
registrar_reembolso    Registra o reembolso de um pedido. Ação irreversível: informe o motivo (ex.: item_faltando, cobranca).
solicitar_reenvio      Solicita o reenvio de um item que veio errado.
escalar_para_humano    Encaminha o atendimento para um atendente humano, com um resumo do caso.
encaminhar_seguranca   Encaminha um relato de golpe ou fraude à equipe de segurança. Não gera compensação.

o que o ADK envia ao modelo sobre buscar_pedido:
{
 "description": "Consulta um pedido pelo número. Devolve itens, valores, pagamentos e atraso.",
 "name": "buscar_pedido",
 "parameters": {
  "pro

In [10]:
# 🎯 EXPERIMENTE: chame uma ferramenta você mesmo, como o ADK faria quando o modelo pede
# ferramentas.consultar_politica("atraso")

Agora o atendimento. As ferramentas agem num banco de pedidos e reembolsos; começamos com ele limpo.

In [11]:
Armazem.reset()  # banco de pedidos e reembolsos limpo para esta seção
resultado = atender_caso("C01", com=com_ferramentas, replay="N1.2/correcao")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
resumo(resultado)

reembolsos = [c.args.get("valor") for c in resultado.trajetoria if c.ferramenta == "registrar_reembolso"]
print("consultou o pedido antes de agir?", chamou(resultado.trajetoria, "buscar_pedido"))
print("reembolsos registrados:", reembolsos, "| preço do refrigerante no pedido: R$ 8,00")

agente,ferramenta,argumentos,efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 5, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 10, 'motivo': 'qualidade'}",✅ com efeito


consultou o pedido antes de agir? False
reembolsos registrados: [5] | preço do refrigerante no pedido: R$ 8,00


**O prompt pede, o modelo não obedece, e o código não percebe.** O prompt ganhou a frase *"Consulte os fatos com as ferramentas antes de decidir. Nunca invente valores de pedido."* e o agente tem `buscar_pedido`. Mesmo assim, nenhuma das 6 execuções gravadas consultou o pedido, e na gravação usada em `replay` o modelo registrou R$ 5 de reembolso por um refrigerante de R$ 8. Ter ferramentas disponíveis **não garante** que o modelo as use. Isso muda nas próximas seções: com a decisão estruturada (N1.4) e o estado da conversa (N1.5), a versão de falha da N1.6 consultou o pedido em 5 de 6 execuções; com o contexto enxuto, em 6 de 6. Com 6 amostras, não dá para separar o efeito de cada componente.

## N1.3 — O loop por baixo

Um agente é um loop: o modelo pede uma ferramenta, o sistema a executa e devolve o resultado, o modelo decide o próximo passo ou responde. `anatomia` mostra esse vaivém a partir dos eventos do ADK, na mesma resposta gravada da N1.2.

In [12]:
from rangoja.observabilidade import anatomia

com_ferramentas = v0_so_prompt.com("ferramentas")
Armazem.reset()  # banco de pedidos e reembolsos limpo para esta seção
resultado = atender_caso("C01", com=com_ferramentas, replay="N1.2/correcao")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
print(anatomia(resultado))

atendimento C01 (v0_so_prompt + ferramentas) — status: concluido
├─ agente unico
│   ├─ pede ferramenta registrar_reembolso({'pedido_id': '4821', 'valor': 5, 'motivo': 'item_faltando'})
│   ├─ pede ferramenta oferecer_cupom({'pedido_id': '4821', 'valor': 10, 'motivo': 'qualidade'})
│   ├─ resultado de registrar_reembolso: {'reembolso_id': 4, 'valor': 5, 'status': 'registrado'}
│   ├─ resultado de oferecer_cupom: {'cupom_id': 1, 'valor': 10, 'status': 'oferecido'}
│   └─ texto: 'Seu pedido já foi processado. Registramos um reembolso de R$ 5 pelo refrigerante que falto'
└─ fonte das respostas do modelo: gravacao


## N1.4 — Prosa × estrutura

Caso `C02` (atraso). Até aqui o agente decide em **prosa**: o código não tem como extrair a categoria e o valor da decisão. A correção é um **contrato**: a decisão sai num formato fixo (`Decisao`), e a mensagem ao cliente vai dentro dela, em texto. O que muda:

In [13]:
sem_schema = v0_so_prompt.com("ferramentas")
com_schema = sem_schema.com("schema")
comparar(sem_schema, com_schema)

In [14]:
Armazem.reset()
em_prosa = atender_caso("C02", com=sem_schema, replay="N1.4/falha")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
resumo(em_prosa)
print("decisão que o código recebe:", em_prosa.decisao)

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10, 'motivo': 'Atraso na entrega'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '5310', 'valor': 7, 'motivo': 'atraso'}",✅ com efeito


decisão que o código recebe: None


Este é o formato que a decisão passa a ter. O ADK o envia ao modelo e valida a resposta contra ele:

In [15]:
from rangoja.politica import solucoes_permitidas
from rangoja.tipos import Decisao
mostrar(Decisao)

Armazem.reset()
estruturada = atender_caso("C02", com=com_schema, replay="N1.4/correcao")
resumo(estruturada)
d = estruturada.decisao
if d is not None:
    print("categoria:", d.categoria, "| solução:", d.solucao, "| valor:", d.valor)
esperar(estruturada, d is not None and d.solucao in solucoes_permitidas(d.categoria),
         "o código consegue conferir a decisão: a solução existe na política para a categoria")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10, 'motivo': 'atraso na entrega'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '5310', 'valor': 7, 'motivo': 'atraso'}",✅ com efeito


categoria: atraso | solução: voucher_e_reembolso_taxa | valor: 10.0


In [16]:
# 🎯 EXPERIMENTE: o schema recusa o que está fora do contrato. Esta solução não existe na política:
# try:
#     Decisao.model_validate({"categoria": "atraso", "solucao": "pix_na_hora", "valor": 10, "pedido_id": "4822"})
# except Exception as erro:
#     print(erro)

Nota: no ADK, a saída estruturada chega por uma ferramenta interna (`set_model_response`). Ela aparece nos eventos, mas não conta como ferramenta do atendimento.

## N1.5 — Estado da conversa

Caso `C11`: três mensagens do mesmo cliente. Sem estado, cada mensagem abre uma sessão nova, e o agente esquece o que veio antes. Aqui a mudança não está no agente, e sim em como a conversa roda:

In [17]:
sem_estado = v0_so_prompt.com("ferramentas", "schema")
com_estado = sem_estado.com("estado")
comparar(sem_estado, com_estado)

conversa = caso("C11")
for i, mensagem in enumerate(conversa.atendimentos[0], 1):
    print(f"mensagem {i}: {mensagem}")

mensagem 1: Meu pedido chegou incompleto.
mensagem 2: É o 4822, faltou o refri.
mensagem 3: Ah, e faltou a sobremesa também.


In [18]:
Armazem.reset()
esquece = atender_caso("C11", com=sem_estado, replay="N1.5/falha")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
resumo(esquece)
print("o que a última sessão sabe:", {k: v for k, v in esquece.estado.items() if k in ("pedido_id", "compensacoes")})

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4822'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso'}",✅ com efeito
unico,escalar_para_humano,"{'motivo': 'Recorrência de reembolsos', 'resumo': 'Cliente solicitou reembolso por item faltante (refrigerante 2L) e já possui 3 reembolsos nos últimos 30 dias.'}",✅ com efeito


o que a última sessão sabe: {}


In [19]:
Armazem.reset()
lembra = atender_caso("C11", com=com_estado, replay="N1.5/correcao")
resumo(lembra); amostra("N1.5/correcao")
print("o que a sessão sabe:", {k: v for k, v in lembra.estado.items() if k in ("pedido_id", "compensacoes")})
esperar(lembra, lembra.estado.get("pedido_id") == "4822", "com estado, a sessão guardou o pedido da conversa")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4822'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso na entrega'}",✅ com efeito


o que a sessão sabe: {'pedido_id': '4822', 'compensacoes': 18.0}


## N1.6 — O que entra no contexto

Caso `C01` de novo. O agente já tem ferramentas, decisão estruturada e estado, mas `buscar_pedido` devolve **o pedido inteiro**, inclusive os dados do entregador, que o modelo não precisa para decidir. A correção é o **contexto enxuto**:

In [20]:
sem_enxuto = v0_so_prompt.com("ferramentas", "schema", "estado")
enxuto = sem_enxuto.com("contexto_enxuto")
comparar(sem_enxuto, enxuto)

In [21]:
Armazem.reset()
completo = atender_caso("C01", com=sem_enxuto, replay="N1.6/falha")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
resumo(completo); amostra("N1.6/falha")
buscas = [c for c in completo.trajetoria if c.ferramenta == "buscar_pedido"]
if buscas:
    print("o que o modelo recebeu sobre o entregador:", buscas[0].resultado.get("entregador"))
else:
    print("nesta execução o modelo não consultou o pedido")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 0, 'motivo': 'item_faltando'}",⛔ sem efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'entrega de item com qualidade abaixo do esperado'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito


o que o modelo recebeu sobre o entregador: {'entregador_id': '301', 'nome_completo': 'Ricardo Fictício Souza', 'telefone': '(00) 90000-1111', 'placa': 'ABC1D23'}


**A resposta está certa, e é por isso que o problema passa despercebido.** O dado do entregador não aparece na resposta, mas está no contexto: o modelo pode repeti-lo a qualquer momento. Em N2.3 simulamos esse vazamento (nas 10 execuções reais, o modelo não vazou) e vemos o guardrail de saída barrá-lo. Aqui a correção vem antes: a ferramenta devolve só o que a decisão precisa.

In [22]:
Armazem.reset()
so_o_necessario = atender_caso("C01", com=enxuto, replay="N1.6/correcao")
resumo(so_o_necessario); amostra("N1.6/correcao")
buscas = [c for c in so_o_necessario.trajetoria if c.ferramenta == "buscar_pedido"]
if buscas:
    print("campos que o modelo recebeu:", sorted(buscas[0].resultado.keys()))
    afirmar("entregador" not in buscas[0].resultado, "com o contexto enxuto, nenhum dado do entregador chega ao modelo")
else:
    print("nesta execução o modelo não consultou o pedido")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
unico,consultar_politica,{'tema': 'item_faltando'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'pedido chegou frio e item faltou'}",✅ com efeito


campos que o modelo recebeu: ['atraso_minutos', 'itens', 'pagamentos', 'pedido_id', 'restaurante', 'status', 'taxa_entrega', 'total']


In [23]:
# 🎯 EXPERIMENTE: o que cada variante de buscar_pedido devolve para outro pedido, sem passar pelo modelo
# from rangoja.ferramentas import _completo, _resumo_enxuto
# print("completo:", sorted(_completo(dados.pedido("5310"))))
# print("enxuto:  ", sorted(_resumo_enxuto(dados.pedido("5310"))))

## N1.7 — Lembrar do cliente

Caso `C13`: dois atendimentos do mesmo cliente, em dias diferentes. No primeiro ele diz que prefere a entrega na portaria. Sem memória, o segundo atendimento não sabe disso. A correção é uma **memória do cliente** (estado com o prefixo `user:`, que atravessa as sessões):

In [24]:
sem_memoria = v0_so_prompt.com("ferramentas", "schema", "estado", "contexto_enxuto")
com_memoria = sem_memoria.com("memoria_livre")  # esta é a v1_agente
comparar(sem_memoria, com_memoria)

cliente = caso("C13")
for i, mensagens in enumerate(cliente.atendimentos, 1):
    print(f"atendimento {i}: {mensagens}")

atendimento 1: ['Pedido 7730 ok, mas prefiro que deixem na portaria.']
atendimento 2: ['Como vai ser a entrega do próximo?']


In [25]:
Armazem.reset()
esquece = atender_caso("C13", com=sem_memoria, replay="N1.7/falha")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
resumo(esquece)
print("o segundo atendimento menciona a portaria?", "portaria" in (esquece.resposta or "").lower())

o segundo atendimento menciona a portaria? False


A memória entra por uma ferramenta nova, `anotar_observacao`: o modelo decide o que anotar, e o texto é gravado em `user:observacoes`, que volta no prompt do próximo atendimento (`{user:observacoes?}`). É isto que o modelo recebe sobre ela:

In [26]:
from google.adk.tools import FunctionTool
print(FunctionTool(ferramentas.anotar_observacao)._get_declaration().model_dump_json(exclude_none=True, indent=1))

{
 "description": "Anota uma observação livre sobre o cliente para atendimentos futuros. Use com cuidado.",
 "name": "anotar_observacao",
 "parameters": {
  "properties": {
   "texto": {
    "type": "STRING"
   }
  },
  "required": [
   "texto"
  ],
  "type": "OBJECT"
 }
}


In [27]:
Armazem.reset()
lembra = atender_caso("C13", com=com_memoria, replay="N1.7/correcao")
resumo(lembra); amostra("N1.7/correcao")
print("observações guardadas para o cliente:", lembra.estado.get("user:observacoes"))
esperar(lembra, "portaria" in (lembra.resposta or "").lower(), "com memória, o segundo atendimento usa a preferência")

agente,ferramenta,argumentos,efeito
unico,consultar_politica,{'tema': 'entrega'},⛔ sem efeito
unico,anotar_observacao,{'texto': 'Cliente preferiu que a entrega fosse feita na portaria.'},✅ com efeito


observações guardadas para o cliente: ['Preferência de entrega na portaria do cliente.', 'Cliente preferiu que a entrega fosse feita na portaria.']


A taxa acima importa: com memória, o modelo **pode** usar a preferência, mas fez isso em poucas das execuções reais. E **memória livre é uma porta de entrada**: no NB02 vemos o cliente usá-la para gravar um direito que não tem.

## N1.8 — Recapitulação

| Problema | Componente | Onde |
|---|---|---|
| O modelo inventa fatos | `ferramentas` | `rangoja/ferramentas.py` |
| Decisão em prosa | `schema` | `rangoja/tipos.py` (`Decisao`) |
| A conversa se perde | `estado` | `session.state` |
| Contexto cheio de dados que não servem | `contexto_enxuto` | `ferramentas._resumo_enxuto` |
| O agente não lembra do cliente | `memoria_livre` | `user:observacoes` |

**Ponte para o NB02:** o agente agora age e lembra. Nada ainda o impede de agir mal.

In [28]:
# 🎯 EXPERIMENTE: tudo o que o NB01 acrescentou, de uma vez
# comparar(v0_so_prompt, v1_agente)